In [ ]:
import os
from os import path
import pickle
import numpy as np
import shutil
from tqdm import tqdm

from macaquethings.plotting.default_styles import *
from macaquethings.data_util.load_data import get_channel_masks
from macaquethings.plotting.anatomical import plot_data_on_anatomy

import matplotlib.pyplot as plt
import seaborn as sns

figure_style()  # set consistent plotting defaults for all figs

## Load Reliability Information

In [ ]:
# specify general options
monkey = 'monkeyN'  # use for the whole script
sessions = "0_3_4_5" if monkey == "monkeyN" else "0_1_2_3_4_5"  # all sessions
label = 'filenames'  # category or filenames

# NOTE ON ARRAYS
# contrary to all other analyses, because each channel is independent here oracle correlation is computed for all electrodes in all arrays
# for consistency with later analyses the plots generated here explicitly exclude array 6, since it is reported to be broken and is also excluded
# in other analyses

single_session = False # if true use session variable below to select oracle correlations for a single session
session = 5      
mua = False  # if true use mua results, else lfp
no_threshold = False  # if false, use all channels, if true use dprime threshold 1.5 consistent with other analyses

if single_session:
    sessions = str(session)

cfg_overrides = {
    'good_channel_threshold': 1.5  # oracle corrs were computed for all channels, apply same threshold as for other analyses
}

# --- derived from above
# oracle_dir = path.join('..', '..', 'results', 'oracle_electrode', f'{monkey}_lfp')
oracle_dir = path.join('..', '..', 'results', 'oracle_electrode', f'{monkey}')

if mua:
    oracle_dir += '_mua'

oracle_file = f'{monkey}-labels_{label}-sessions_{sessions}-rois_1_2_3-arrays_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16-baseline_0-standardize_1.pkl'


# --- set up savedir
if not single_session:
    savedir = 'electrode_reliability_panels'
else:
    savedir = f'electrode_reliability_panels_{sessions}'

if mua:
    savedir += "_mua"
    
os.makedirs(savedir, exist_ok=True)

# --- colors


if monkey == 'monkeyF':
    array_to_roi = {
        1: 'v1',
        2: 'v1',
        3: 'v1',
        4: 'v1',
        5: 'v1',
        6: 'v1',
        7: 'v1',
        8: 'v1',
        9: 'it',
        10: 'it',
        11: 'it',
        12: 'it',
        13: 'it',
        14: 'v4',
        15: 'v4',
        16: 'v4',
    }
else:
    array_to_roi = {
        1: 'v1',
        2: 'v1',
        3: 'v1',
        4: 'v1',
        5: 'v1',
        6: 'v1',
        7: 'v1',
        8: 'v1',
        9: 'v4',
        10: 'v4',
        11: 'v4',
        12: 'v4',
        13: 'it',
        14: 'it',
        15: 'it',
        16: 'it',
    }    

roi_to_color = {
    'v1': 'tab:blue',
    'v4': 'tab:orange',
    'it': 'tab:green'
}

array_to_color = {}

n_v1 = (np.array(list(array_to_roi.values())) == 'v1').sum()
n_v4 = (np.array(list(array_to_roi.values())) == 'v4').sum()
n_it = (np.array(list(array_to_roi.values())) == 'it').sum()

blues = sns.color_palette('Blues', n_v1 + 1)[1:]
oranges = sns.color_palette('Oranges', n_v4 + 1)[1:]
greens = sns.color_palette('Greens', n_it + 1)[1:]

cmaps_per_roi = {
    'v1': blues,
    'v4': oranges,
    'it': greens
}

v1_counter = 0
v4_counter = 0
it_counter = 0

for arr in array_to_roi.keys():
    roi = array_to_roi[arr]
    cmap = cmaps_per_roi[roi]
    if roi == 'v1':
        color = cmap[v1_counter]
        v1_counter += 1
    elif roi == 'v4':
        color = cmap[v4_counter]
        v4_counter += 1
    else:
        color = cmap[it_counter]
        it_counter += 1
    array_to_color[arr] = color


from matplotlib.lines import Line2D
import matplotlib.pyplot as plt

def create_legend(array_to_color, linestyles):
    """
    Create a custom legend:
    - One entry per array color
    - One entry per linestyle (black)
    
    Parameters:
        array_to_color : dict
            Mapping of array index -> color
        linestyles : dict
            Mapping of label type -> linestyle (e.g., {'label1':'-', 'label2':'--'})
    """
    legend_elements = []

    # One entry per array (color)
    for arr, color in array_to_color.items():
        legend_elements.append(Line2D([0], [0], color=color, lw=2, label=f'Array {arr}'))

    # One entry per linestyle (black)
    for label_type, linestyle in linestyles.items():
        legend_elements.append(Line2D([0], [0], color='black', lw=2, linestyle=linestyle, label=label_type))

    # Create figure for the legend only
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.legend(handles=legend_elements, loc='center', ncol=4, frameon=False)
    ax.axis('off')  # hide axes
    plt.show()

create_legend(array_to_color, linestyles={'category': '-', 'stimulus identity': '--'})
plt.savefig(path.join(savedir, f'{monkey}_full_array_legend.svg'))

In [ ]:
with open(path.join(oracle_dir, oracle_file), 'rb') as f:
    oracle_results = pickle.load(f)

oracle_corr = oracle_results['oracle'].T  # for this script convention is to have time first
# oracle_time = oracle_results['oracle_cfg']['time_select'].mean(axis=1)
oracle_time = oracle_results['time']

data_cfg = oracle_results["data_cfg"]
data_cfg.update(cfg_overrides)
oracle_masks = get_channel_masks(data_cfg, root='../..')

oracle_rois = oracle_masks['rois']
oracle_arrays = oracle_masks['array_ids']
good_channels = oracle_masks['good_channels']

# if monkeyN, make sure we exclude all channels for array 6
if monkey == 'monkeyN':
    good_channels[oracle_arrays == 6] = False

if no_threshold:
    good_channels = np.ones(1024).astype(bool)


# Plot for all good channels

In [ ]:
plt.figure(figsize=HALF_PANEL_SIZE)
plt.plot(oracle_time, oracle_corr[:, good_channels], color='k', alpha=0.05, linewidth=1.)
plt.xlabel('time (ms)')
plt.ylabel('oracle correlation')
plt.title(f"{monkey} - {label} - oracle correlation")
plt.savefig(
    path.join(savedir, f"{monkey}_{label}_allchannels.svg")
)

# Single Panel, Color by ROI

In [ ]:
xlim = (0, 300)

unique_rois = np.sort(np.unique(oracle_rois))
plt.figure(figsize=(THIRD_WIDTH, THIRD_WIDTH / 2))
for ur in unique_rois:
    mask = (oracle_rois == ur) & good_channels
    oracles_select = oracle_corr[:, mask]
    roiname = {1: 'v1', 2: 'v4', 3: 'it'}[ur]
    color = roi_to_color[roiname]
    plt.plot(oracle_time, oracles_select, color=color, alpha=.01, linewidth=1)
    plt.plot(oracle_time, oracles_select.mean(axis=1), color=color, label=roiname, alpha=1.)
plt.xlim(xlim)
plt.xlabel('time (ms)')
plt.ylabel('oracle correlation')
plt.title(f"{monkey} - {label} - oracle correlation")
plt.legend()
plt.savefig(
    path.join(savedir, f"{monkey}_{label}_per_roi.svg")
)


# One Panel per ROI

In [ ]:
unique_rois = np.sort(np.unique(oracle_rois))
for ur in unique_rois:
    plt.figure(figsize=QUARTER_PANEL_SIZE)
    mask = (oracle_rois == ur) & good_channels
    oracles_select = oracle_corr[:, mask]
    roiname = {1: 'v1', 2: 'v4', 3: 'it'}[ur]
    color = roi_to_color[roiname]
    plt.plot(oracle_time, oracles_select, color=color, alpha=.01, linewidth=1)
    plt.plot(oracle_time, oracles_select.mean(axis=1), color=color, label=roiname, alpha=1.)
    plt.xlabel('time (ms)')
    plt.ylabel('oracle correlation')
    plt.title(f"{monkey} - {label} - oracle correlation - {roiname}")
    plt.xlim(xlim)
    plt.legend()
    plt.savefig(
    path.join(savedir, f"{monkey}_{label}_{roiname}.svg")
    )

# Panel per ROI, Color by Array

In [ ]:
unique_rois = np.sort(np.unique(oracle_rois))
for ur in unique_rois:
    plt.figure(figsize=QUARTER_PANEL_SIZE)
    arrs_in_roi = np.sort(np.unique(oracle_arrays[oracle_rois == ur]))
    # --- select channels for ROI                  
    mask = (oracle_rois == ur) & good_channels
    oracles_select = oracle_corr[:, mask]
    array_mask_select = oracle_arrays[mask]
    roiname = {1: 'v1', 2: 'v4', 3: 'it'}[ur]
    # --- select channels per array
    for i, ua in enumerate(arrs_in_roi):
        color = cmaps_per_roi[roiname][i]
        oracles_arr = oracles_select[:, array_mask_select == ua]
        # --- all channels
        plt.plot(oracle_time, oracles_arr, color=color, alpha=.05, linewidth=.5)
        # --- mean and label
        plt.plot(oracle_time, oracles_arr.mean(axis=1), color=color, alpha=1., label=ua)
        
    plt.xlabel('time (ms)')
    plt.ylabel('oracle correlation')
    plt.title(f"{monkey} - {label} - oracle correlation - {roiname}")
    plt.xlim(xlim)
    plt.savefig(
    path.join(savedir, f"{monkey}_{label}_{roiname}_perarray.svg")
    ) 

# Plot on anatomy

In [ ]:
oracle_corr_masked = oracle_corr.copy()
oracle_corr_masked[:, ~ good_channels] = np.nan

vmin = np.nanmin(oracle_corr_masked)
vmax = np.nanmax(oracle_corr_masked)

if not single_session:
    movie_dir = 'electrode_reliability_movies'
else:
    movie_dir = f'electrode_reliability_movies_{sessions}'

if mua:
    movie_dir += "_mua"
    
os.makedirs(movie_dir, exist_ok=True)

tmp_framedir = path.join(movie_dir, 'tmp')

In [ ]:
# create a subdir for frames

# if exists, delete
if path.isdir(tmp_framedir):
    shutil.rmtree(tmp_framedir)

os.makedirs(tmp_framedir)

for i, t in tqdm(enumerate(oracle_time)):
    plot_data_on_anatomy(monkey, oracle_corr_masked[i], vmin=vmin, vmax=vmax, root='../..')
    t = str(int(t)).zfill(4)
    plt.title(f"{t} ms")
    plt.savefig(path.join(tmp_framedir, f'{t}.png'), dpi=300)
    plt.close(plt.gcf())

In [ ]:
# create the movie
fpath = path.join(movie_dir, f"{monkey}_reliability_movie.mp4")

!ffmpeg -framerate 30 -i $tmp_framedir/%04d.png -c:v mpeg4 -q:v 2 -pix_fmt yuv420p $fpath -y -threads 8

In [ ]:
def threshold_crossings(data, frac=0.5):
    """
    data: numpy array of shape (time, channels)
    frac: fraction of the channel max to exceed (default 0.5)
    """
    # max per channel
    max_vals = data.max(axis=0)
    
    # thresholds per channel
    thresholds = frac * max_vals

    # Boolean mask: True where value exceeds threshold
    exceeds = data > thresholds

    # For each channel, find first index where it exceeds the threshold
    # np.argmax returns 0 if all False, so we need to detect channels that never exceed
    idx = exceeds.argmax(axis=0)

    # Fix channels that never cross threshold: if no True exists, all values = False
    never_cross = ~exceeds.any(axis=0)
    idx[never_cross] = -1   # or np.nan if preferred

    return idx


# Create a writable copy of the magma colormap
cmap = sns.color_palette('rocket', as_cmap=True)
cmap.set_over('yellow')      # values > vmax
cmap.set_under('cyan')    # values < vmin

threshold = .5
first_idx_above_threshold = threshold_crossings(oracle_corr, threshold)
first_time_above_threshold = oracle_time[first_idx_above_threshold].astype(float)
first_time_above_threshold[~ good_channels] = np.nan
_, cbar = plot_data_on_anatomy(monkey, first_time_above_threshold, root='../..', return_cbar=True, vmin=0, vmax=120, cmap_name=cmap)
cbar.set_label('time (ms)')
plt.title(f"{monkey} - time to reliability > {threshold} of max")
plt.savefig(
    path.join(savedir, f"{monkey}_{label}_anatomy_time_to_{threshold}_of_max.svg")
)  
plt.show()